In [ ]:
import pandas as pd
import requests
import time
import os
from geopy.geocoders import Nominatim


In [ ]:
# 1. Load the dataset
df = pd.read_csv('../data/processed/focused_crop_data.csv')
unique_locations = df[['State', 'District']].drop_duplicates().reset_index(drop=True)

geolocator = Nominatim(user_agent="agroplan_x_weather_bot")
weather_records = []

def get_coordinates(state, district):
    query = f"{district}, {state}, India"
    try:
        location = geolocator.geocode(query, timeout=10)
        if location:
            return location.latitude, location.longitude
    except Exception as e:
        print(f"Geocode error for {district}: {e}")
    return None, None

def get_nasa_weather(lat, lon, start="2022", end="2025"):
    url = f"https://power.larc.nasa.gov/api/temporal/monthly/point?parameters=T2M,PRECTOTCORR&community=AG&longitude={lon}&latitude={lat}&start={start}&end={end}&format=JSON"
    try:
        response = requests.get(url)
        if response.status_code == 200:
            return response.json()['properties']['parameter']
    except Exception as e:
        print(f"NASA API error for {lat},{lon}: {e}")
    return None

print(f"Starting download for {len(unique_locations)} districts. This will take ~15 minutes...")

# 2. Iterate through all districts
for index, row in unique_locations.iterrows():
    state = row['State']
    district = row['District']
    
    lat, lon = get_coordinates(state, district)
    
    if lat and lon:
        data = get_nasa_weather(lat, lon)
        if data:
            # NASA data comes as { 'YYYYMM': value }. We flatten this into rows.
            for year_month, rain_val in data['PRECTOTCORR'].items():
                temp_val = data['T2M'].get(year_month)
                
                weather_records.append({
                    'State': state,
                    'District': district,
                    'YearMonth': year_month, # e.g., '202201'
                    'Rainfall_mm': rain_val,
                    'Temperature_C': temp_val
                })
        print(f"Success: {district} ({index+1}/{len(unique_locations)})")
    else:
        print(f"Failed to find coordinates for: {district}")
        
    time.sleep(1) # Be polite to the APIs!

# 3. Save the raw weather data
weather_df = pd.DataFrame(weather_records)

Starting download for 717 districts. This will take ~15 minutes...
Success: North And Middle Andaman (1/717)
Success: Alluri Sitharama Raju (2/717)
Success: Anakapalli (3/717)
Success: Anantapur (4/717)
Success: Annamayya (5/717)
Success: Bapatla (6/717)
Success: Chittoor (7/717)
Success: Dr. B.R. Ambedkar Konaseema (8/717)
Success: East Godavari (9/717)
Success: Eluru (10/717)
Success: Guntur (11/717)
Success: Kakinada (12/717)
Success: Krishna (13/717)
Success: Kurnool (14/717)
Success: Nandyal (15/717)
Success: Ntr (16/717)
Success: Palnadu (17/717)
Success: Parvathipuram Manyam (18/717)
Success: Prakasam (19/717)
Success: Spsr Nellore (20/717)
Success: Sri Sathya Sai (21/717)
Success: Srikakulam (22/717)
Success: Tirupati (23/717)
Failed to find coordinates for: Visakhapatanam
Success: Vizianagaram (25/717)
Success: West Godavari (26/717)
Success: Y.S.R. (27/717)
Success: Anjaw (28/717)
Success: Changlang (29/717)
Success: Dibang Valley (30/717)
Success: East Kameng (31/717)
Succes